# Hands-on: Forecast the Weather with a Recurrent Network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/RNN_colab.ipynb)

This notebook lets you **train the weather-forecasting RNN yourself**, compare it with simple baselines,
and try it on **your own location**. The ideas behind it (hidden state, LSTM gates, baselines) are explained
in the presenter's [marimo notebook](https://molab.marimo.io/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/RNN.py).

**No coding needed:** run each cell in order with the ▶ button (or `Shift + Enter`). Cells marked
**✏️ Settings** have a few values you can change before running them.

**Tip:** training is fast even on CPU; a GPU (*Runtime → Change runtime type → T4 GPU*) makes long
histories faster.

## 1. Setup
Downloads the shared workshop code and two years of hourly weather for Chachapoyas (NASA POWER).

In [ ]:
import os, urllib.request
if not os.path.exists("rnn_common.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/lobaton/AI-Workshop-UNTRM/main/session2_inside_the_black_box/src/rnn_common.py", "rnn_common.py")

import numpy as np
import matplotlib.pyplot as plt
import torch
import rnn_common as rc

plt.rcParams.update({"font.size": 14, "lines.linewidth": 3})
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", "GPU" if device == "cuda" else "CPU")

weather = rc.load_weather(rc.fetch(rc.WEATHER_FILE))
print(f"{len(weather)} hours from {weather['time'].iloc[0]:%Y-%m-%d} to {weather['time'].iloc[-1]:%Y-%m-%d}")
weather.head()

## 2. ✏️ Settings

- `TARGET`: what to forecast, `"temperature"` (°C) or `"humidity"` (%).
- `INPUTS`: which variables the network reads: any of `"temperature"`, `"humidity"`, `"rain"`, `"wind"`,
  `"pressure"` (time of day is always added).
- `HISTORY_HOURS`: how many past hours the network sees. `HORIZON_HOURS`: how far ahead to forecast (1–24).
- `CELL`: `"LSTM"`, `"GRU"` or `"RNN"` (a plain RNN without gates).
- `HIDDEN`: number of hidden units. `EPOCHS`: passes through the training data.

In [ ]:
TARGET = "temperature"
INPUTS = ["temperature", "humidity", "rain"]
HISTORY_HOURS = 48
HORIZON_HOURS = 6
CELL = "LSTM"
HIDDEN = 32
EPOCHS = 20

## 3. Train and compare with the baselines

The first 70% of the record is used for training, the next 15% for validation and the last 15% for the
final test. The table compares the network with two free guesses: **persistence** ("same as now") and
**same hour yesterday**.

In [ ]:
def train_and_compare(weather, label):
    fc = rc.Forecaster(weather, TARGET, INPUTS, HISTORY_HOURS, HORIZON_HOURS)
    train, val, test = fc.windows("train"), fc.windows("validation"), fc.windows("test")
    torch.manual_seed(0)
    model = rc.WeatherRNN(len(fc.inputs), HIDDEN, CELL)
    history = rc.train_model(model, fc, train, val, epochs=EPOCHS, device=device)
    pred = rc.predict(model, test["X"], fc)
    unit = rc.VARIABLES[TARGET][2]
    print(f"{label}: trained {CELL} in {history['seconds']:.0f} s. Test mean absolute error ({unit}):")
    print(f"  Persistence:         {rc.mae(test['persistence'], test['y']):.2f}")
    print(f"  Same hour yesterday: {rc.mae(test['yesterday'], test['y']):.2f}")
    print(f"  {CELL}:{' ' * (19 - len(CELL))} {rc.mae(pred, test['y']):.2f}")
    return fc, model, history, test, pred

fc, model, history, test, pred = train_and_compare(weather, "Chachapoyas")

plt.figure(figsize=(9, 4))
plt.plot(history["train_mae"], "o-", label="Training")
plt.plot(history["val_mae"], "s--", label="Validation")
plt.xlabel("Epoch"); plt.ylabel(f"MAE ({rc.VARIABLES[TARGET][2]})"); plt.title("Learning curve")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Try:**
- `CELL = "RNN"` vs. `"LSTM"`: does the gated network help?
- `HISTORY_HOURS = 24` vs. `120`: does more history always help?
- `HORIZON_HOURS = 1` vs. `24`: when does the network beat "same hour yesterday" by the most?

## 4. Look at some forecasts
Each panel is one test example: the history the network read, what really happened, and the forecasts.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, i in zip(axes.ravel(), np.linspace(100, len(test["y"]) - 100, 4).astype(int)):
    now = int(test["now"][i])
    hours = np.arange(-HISTORY_HOURS + 1, HORIZON_HOURS + 1)
    series = weather[TARGET].values[now - HISTORY_HOURS + 1:now + HORIZON_HOURS + 1]
    ax.plot(hours, series, color="gray", label="Observed")
    ax.axvline(0, color="black", linestyle=":")
    ax.plot(HORIZON_HOURS, test["y"][i], "o", color="black", markersize=10, label="True")
    ax.plot(HORIZON_HOURS, pred[i], "*", color="#009E73", markersize=18, label=CELL)
    ax.plot(HORIZON_HOURS, test["yesterday"][i], "D", color="#E69F00", markersize=9, label="Yesterday")
    ax.set_title(f"{weather['time'].iloc[now]:%Y-%m-%d %H:%M}", fontsize=13)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=11)
fig.supxlabel("Hours relative to now"); fig.supylabel(rc.VARIABLES[TARGET][1])
plt.tight_layout(); plt.show()

## 5. ✏️ Your own location

Change `LAT` and `LON` to any place (for example, a farm, or another town in Amazonas) and run the next two
cells. The weather is downloaded live from NASA POWER (about 1 minute) and the same network is trained on
it with the settings from Section 2.

*Find coordinates by right-clicking a place in Google Maps. South latitudes and west longitudes are
negative.* NASA POWER estimates weather for an area of about 50 km, so it is smoother than a local weather
station.

In [ ]:
LAT, LON = -5.74, -78.44   # Example: Bagua Grande, Amazonas
PLACE = "Bagua Grande"

In [ ]:
my_weather = rc.fetch_nasa_power(LAT, LON, "20230101", "20241231")
hour = my_weather["time"].dt.hour
my_weather["hour_sin"] = np.sin(2 * np.pi * hour / 24)
my_weather["hour_cos"] = np.cos(2 * np.pi * hour / 24)
print(f"Average temperature: {my_weather['temperature'].mean():.1f} °C "
      f"(Chachapoyas: {weather['temperature'].mean():.1f} °C)")
_ = train_and_compare(my_weather, PLACE)

*Have data from your own weather station?* Upload a CSV with a `time` column and columns named like
the ones above (`temperature`, `humidity`, `rain`, ...), load it with `rc.load_weather("your_file.csv")`
and call `train_and_compare` on it.

## Questions for discussion

1. Was the network's error at your location larger or smaller than in Chachapoyas? What about the
   location's weather could explain it?
2. For a coffee farmer worried about leaf rust, which would be more useful: a very accurate forecast 1 hour
   ahead, or a less accurate one 24 hours ahead? Why?